In [13]:
# NetCDF Metadata Editor - Jupyter Notebook
# Run each cell sequentially to view and edit netCDF metadata
# Tested with the FLUXNET2015 dataset for the REF

# Cell 1: Import libraries
import netCDF4 as nc
import shutil
import os
from pathlib import Path

print("Libraries imported successfully!")

Libraries imported successfully!


In [14]:
# ============================================================
# Cell 2: Configure your file and settings
# ============================================================

# Define your file path here
file_path = "/obs4MIPs-cmor-tables/inputs/PCMDI/ORNL/FLUXNET2015-1-0/gpp_mon_Fluxnet-2015-1-0_REF_site_19910101-20141231.nc" # <-- CHANGE THIS to your file path

# Define the attribute you want to edit
attribute_name = "source_label"  # <-- CHANGE THIS to the attribute name

# Define the new value
new_value = "FLUXNET"  # <-- CHANGE THIS to your desired value

print(f"Configuration set:")
print(f"  File: {file_path}")
print(f"  Attribute: {attribute_name}")
print(f"  New value: {new_value}")

Configuration set:
  File: /obs4MIPs-cmor-tables/inputs/PCMDI/ORNL/FLUXNET2015-1-0/gpp_mon_Fluxnet-2015-1-0_REF_site_19910101-20141231.nc
  Attribute: source_label
  New value: FLUXNET


In [15]:
# ============================================================
# Cell 3: Check if file exists
# ============================================================

if Path(file_path).exists():
    print(f"\u2713 File found: {file_path}")
else:
    print(f"\u2717 Error: File not found: {file_path}")
    print("Please update the file_path in Cell 2")

✓ File found: /Users/paul.smith/Documents/obs4MIPs-cmor-tables/inputs/PCMDI/ORNL/FLUXNET2015-1-0/gpp_mon_Fluxnet-2015-1-0_REF_site_19910101-20141231.nc


In [16]:
# ============================================================
# Cell 4: View all current metadata
# ============================================================

try:
    with nc.Dataset(file_path, 'r') as ds:
        print("\n=== Current Global Attributes ===\n")
        for attr in ds.ncattrs():
            value = getattr(ds, attr)
            print(f"{attr}: {value}")
        
        # Store current value if attribute exists
        if attribute_name in ds.ncattrs():
            current_value = getattr(ds, attribute_name)
            print(f"\n>>> Current value of '{attribute_name}': {current_value}")
        else:
            print(f"\n>>> Attribute '{attribute_name}' does not exist (will be created)")
            
except FileNotFoundError:
    print(f"Error: Cannot open file '{file_path}'")
except Exception as e:
    print(f"Error reading file: {e}")


=== Current Global Attributes ===

Conventions: CF-1.12 ODS-2.6
aux_uncertainty_id: stderr
comment: gpp=(DT+NT)/2 and gpp_stderr = |DT-NT|/2
contact: Fluxnet Support Team (fluxdata-support@fluxdata.org)
creation_date: 2026-02-02
data_specs_version: 2.6
dataset_contributor: Nathan Collier
doi: N/A
frequency: mon
grid: site
grid_label: site
has_aux_unc: TRUE
history: 
2026-01-28: downloaded using https://fluxnet.org/data/download-data/;
2026-02-02: converted to obs4MIP format
institution: The Fluxnet Community
institution_id: Fluxnet
license: Data in this file produced by ILAMB is licensed under a Creative Commons Attribution - 4.0 International (CC BY 4.0) License (https://creativecommons.org/licenses/).
nominal_resolution: site
processing_code_location: https://github.com/rubisco-sfa/ilamb3-data/blob/main/data/Fluxnet-2015/convert.py
product: site-observations
realm: land
references: Pastorello, Gilberto and Trotta, Carlo and Canfora, Eleonora and Chu, et al., The FLUXNET2015 dataset 

In [17]:
# ============================================================
# Cell 5: Edit the metadata (RUN THIS CELL TO MAKE CHANGES)
# ============================================================
# Uses a copy-and-replace strategy to avoid HDF5 file locking
# errors that occur when opening netCDF4 files in append mode.
# A temporary copy is modified, then atomically replaces the
# original, so the source file is never left in a broken state.
# ============================================================

tmp_path = file_path + ".tmp"

try:
    # Step 1: Copy original to a temporary file
    shutil.copy2(file_path, tmp_path)
    print(f"\u2713 Temporary copy created: {tmp_path}")

    # Step 2: Edit the attribute on the temporary copy
    with nc.Dataset(tmp_path, 'a') as ds:
        if attribute_name in ds.ncattrs():
            old_value = getattr(ds, attribute_name)
            print(f"Changing '{attribute_name}':")
            print(f"  FROM: {old_value}")
            print(f"  TO:   {new_value}")
        else:
            print(f"Creating new attribute '{attribute_name}' with value: {new_value}")

        setattr(ds, attribute_name, new_value)

    # Step 3: Atomically replace the original with the modified copy
    os.replace(tmp_path, file_path)
    print("\n\u2713 Metadata updated successfully!")

except PermissionError:
    if os.path.exists(tmp_path):
        os.remove(tmp_path)
    print("Error: Permission denied. Check file permissions or whether it is open elsewhere.")
except Exception as e:
    if os.path.exists(tmp_path):
        os.remove(tmp_path)
    print(f"Error editing metadata: {e}")

✓ Temporary copy created: /Users/paul.smith/Documents/obs4MIPs-cmor-tables/inputs/PCMDI/ORNL/FLUXNET2015-1-0/gpp_mon_Fluxnet-2015-1-0_REF_site_19910101-20141231.nc.tmp
Changing 'source_label':
  FROM: Fluxnet
  TO:   FLUXNET

✓ Metadata updated successfully!


In [18]:
# ============================================================
# Cell 6: Verify the changes
# ============================================================

try:
    with nc.Dataset(file_path, 'r') as ds:
        print("\n=== Updated Global Attributes ===\n")
        for attr in ds.ncattrs():
            value = getattr(ds, attr)
            # Highlight the changed attribute
            if attr == attribute_name:
                print(f">>> {attr}: {value} <<<")
            else:
                print(f"{attr}: {value}")
                
except Exception as e:
    print(f"Error reading file: {e}")


=== Updated Global Attributes ===

Conventions: CF-1.12 ODS-2.6
aux_uncertainty_id: stderr
comment: gpp=(DT+NT)/2 and gpp_stderr = |DT-NT|/2
contact: Fluxnet Support Team (fluxdata-support@fluxdata.org)
creation_date: 2026-02-02
data_specs_version: 2.6
dataset_contributor: Nathan Collier
doi: N/A
frequency: mon
grid: site
grid_label: site
has_aux_unc: TRUE
history: 
2026-01-28: downloaded using https://fluxnet.org/data/download-data/;
2026-02-02: converted to obs4MIP format
institution: The Fluxnet Community
institution_id: Fluxnet
license: Data in this file produced by ILAMB is licensed under a Creative Commons Attribution - 4.0 International (CC BY 4.0) License (https://creativecommons.org/licenses/).
nominal_resolution: site
processing_code_location: https://github.com/rubisco-sfa/ilamb3-data/blob/main/data/Fluxnet-2015/convert.py
product: site-observations
realm: land
references: Pastorello, Gilberto and Trotta, Carlo and Canfora, Eleonora and Chu, et al., The FLUXNET2015 dataset 

In [ ]:
# ============================================================
# Cell 7 (Optional): View dataset structure
# ============================================================

try:
    with nc.Dataset(file_path, 'r') as ds:
        print("\n=== Dataset Structure ===\n")
        print(f"Dimensions: {list(ds.dimensions.keys())}")
        print(f"Variables: {list(ds.variables.keys())}")
        print(f"\nGlobal Attributes: {len(ds.ncattrs())}")
        
except Exception as e:
    print(f"Error: {e}")